# Day 4 Project — Engineering Design Review Team

## Before you begin

### Learning outcomes

- Assemble the whole day: provider contract, deterministic checks, bounded fan-out, supervisor fan-in, and measurement.
- Turn a quality requirement into a deployment decision the code can make.
- Write a decision memo that a reviewer could disagree with using your own numbers.

Architecture reference: [Day 4 diagrams D12–D15](../diagrams/source/day_04.md)

### Expected observation

All three systems terminate with structured findings and inspectable traces, and the recommended system changes when the quality bar changes.


## Concept briefing

## What to carry into Day 5

Days 1-4 repeatedly configure providers, validate tool and model output, enforce limits,
fall back safely and record events. Day 5 pulls those repeated responsibilities into
reusable infrastructure while keeping application-specific instructions, tools and policy
in agent configuration.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Assemble the system

One provider, three architectures, two scenarios. Nothing new is introduced here — this is the day, wired together.


In [ ]:
from review_team import (FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer,
                         evaluate, run_checks_plus_reviewer, run_single_reviewer,
                         run_specialist_team)

SCENARIOS_TO_RUN = ("blind_spots", "strong_generalist")

def build_provider(scenario):
    if LIVE:
        return FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(scenario))
    return MockStructuredReviewer(scenario)

def run_everything(scenario):
    provider = build_provider(scenario)
    runs = [run_single_reviewer(SOURCE, provider),
            run_checks_plus_reviewer(SOURCE, provider),
            run_specialist_team(SOURCE, provider)]
    return runs, [evaluate(run, GOLDEN_PATH) for run in runs]

results = {scenario: run_everything(scenario) for scenario in SCENARIOS_TO_RUN}

print("Provider:", "live model with mock fallback" if LIVE else "MockStructuredReviewer")
for scenario, (runs, rows) in results.items():
    print("\n" + scenario)
    for row in rows:
        print(f"   {row['system']:<24} found {row['found']}/9 | calls {row['model_calls']} | "
              f"tokens {row['tokens']} | merged {row['merged_duplicates']} | "
              f"dropped {row['dropped_over_cap']}")


## Step 3 — Inspect a trace end to end

Every system must be debuggable by reading, not guessing. This is the full trace of the largest one.


In [ ]:
team_run = results["blind_spots"][0][2]

print("System:", team_run.system, "\n")
for step in team_run.trace:
    print(step["step"])
    for key, value in step.items():
        if key != "step":
            print("   ", key, "=", value)

print("\nFinal report:")
for finding in team_run.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}  "
          f"[{finding.reviewer}]")


## Step 4 — Check the bounds, not the wording

These assertions are about *structure*: the system stayed inside its limits and every claim carries evidence. Quality is measured, never asserted.


In [ ]:
checks_passed = []

for scenario, (runs, rows) in results.items():
    for run, row in zip(runs, rows):
        assert run.model_calls <= 3, "fan-out must stay bounded"
        assert 0.0 <= row["recall"] <= 1.0
        assert row["false_positives"] == 0, "no finding may point outside the artifact"
        assert all(f.evidence.strip() for f in run.findings), "every finding needs evidence"
        assert len(run.findings) <= 20, "the supervisor must cap its report"
        checks_passed.append(f"{scenario}/{run.system}")

print("Structural checks passed for:")
for name in checks_passed:
    print("   ", name)
print("\nNone of these assert that the multi-agent system wins. That is an observation.")


## Step 5 — Turn a requirement into a decision

A deployment choice needs a stated quality bar. Given one, the rule is mechanical: **the smallest system that clears the bar**.


In [ ]:
def recommend(rows, minimum_recall):
    """Smallest system (fewest model calls) whose recall meets the bar."""
    qualifying = [row for row in rows if row["recall"] >= minimum_recall]
    if not qualifying:
        return None
    return min(qualifying, key=lambda row: (row["model_calls"], row["tokens"]))

BAR = 0.85
print(f"Quality bar: recall >= {BAR}\n")
for scenario, (_runs, rows) in results.items():
    choice = recommend(rows, BAR)
    if choice is None:
        print(f"{scenario:<20} no system meets the bar; do not deploy")
    else:
        print(f"{scenario:<20} deploy {choice['system']:<24} "
              f"(recall {choice['recall']}, {choice['model_calls']} call(s), "
              f"{choice['tokens']} tokens)")


### Try it yourself

Raise the quality bar to 1.0 — nothing may be missed. Predict which system gets recommended in each scenario, and whether any recommendation becomes "do not deploy".


In [ ]:
# --- Worked solution ---
for bar in (0.85, 1.0):
    print(f"Quality bar: recall >= {bar}")
    for scenario, (_runs, rows) in results.items():
        choice = recommend(rows, bar)
        verdict = "DO NOT DEPLOY (no system meets the bar)" if choice is None else (
            f"{choice['system']} ({choice['model_calls']} call(s), {choice['tokens']} tokens)")
        print(f"   {scenario:<20} -> {verdict}")
    print()

print("Raising the bar changes the answer, and in strong_generalist it removes every")
print("option: no amount of orchestration finds DEF-COR-02, because neither the general")
print("reviewer nor the correctness specialist can see it. At that point the fix is a")
print("better reviewer, a deterministic check, or a human - not another agent.")


## Step 6 — Your decision memo

One paragraph, using your own numbers from the tables above. The template below prints a filled-in example so you know exactly what is expected.


In [ ]:
rows_blind = results["blind_spots"][1]
single_row, _augmented_row, team_row = rows_blind

memo = f"""DECISION MEMO - Engineering Design Review Team

Chosen system : {team_row['system']} (scenario: blind_spots)
Evidence      : recall {team_row['recall']} ({team_row['found']}/9) versus
                {single_row['recall']} ({single_row['found']}/9) for a single reviewer.
                False positives {team_row['false_positives']}; duplicates surviving
                synthesis {team_row['duplicates']}; the supervisor merged
                {team_row['merged_duplicates']} overlapping findings and dropped
                {team_row['dropped_over_cap']} over its cap.
Cost          : {team_row['model_calls']} model calls and {team_row['tokens']} tokens versus
                {single_row['model_calls']} call and {single_row['tokens']} tokens - about
                {team_row['tokens'] / single_row['tokens']:.1f}x the spend for
                {team_row['found'] - single_row['found']} extra defects.
Latency       : parallel fan-out cut wall clock roughly 3x in notebook 4.4; it did not
                reduce calls or tokens.
Debugging     : 4 branches plus a merge rule instead of 1 call - more places to be wrong.
Reverses if   : the reviewer improves. Measured in the strong_generalist scenario, the
                team found {results['strong_generalist'][1][2]['found']}/9, exactly what one
                reviewer found, for 3x the calls. Then deploy the single reviewer.
"""
print(memo)


### Checkpoint

**1. Your team wants to add a fourth specialist (performance). What must you show before and after adding it?**

<details><summary>Show answer</summary>

The same table, measured both ways: recall, false positives, duplicates, model calls, tokens, latency and the merge report. Adding a branch is justified only if it finds defects the current system misses, at a cost you would still pay knowing the number. The architecture supports it — `SPECIALIST_ROLES` and the supervisor do not change — which is exactly why the discipline has to come from the measurement.

</details>

**2. At a quality bar of 1.0, no system qualifies in the `strong_generalist` scenario. What is the correct engineering response?**

<details><summary>Show answer</summary>

Not "add more agents". The missed defect (DEF-COR-02, a flat discount that can drive a total negative) is invisible to the generalist *and* to the correctness specialist, so it is a capability gap, not an attention gap. The options are a stronger reviewer, a deterministic check or test that encodes the business rule, or a human reviewer for that class of defect.

</details>

### Recap

- Limitation we saw: a system can hit every structural bound and still fail the quality requirement, and more agents will not fix it.
- Layer we added: an explicit quality bar plus a mechanical rule — the smallest system that clears it — and a memo built from measured numbers.
- Evidence it worked: the recommendation flips between scenarios and flips again when the bar moves from 0.85 to 1.0.
